# Small Sample Training (10 Samples, Limited Epochs)
This notebook trains the VibeShift flow matching model on a small sample of data for quick testing and validation.
- **Sample Size**: 10 pairs (latent_classical ↔ latent_synth)
- **Epochs**: 5 (configurable)
- **Use Case**: Testing, debugging, and quick validation before full training

In [1]:
import sys
import os
from pathlib import Path

# Add project root to path
ROOT = Path(r"c:\Users\Dhanuja\Desktop\Vibeshift\VibeShift")
sys.path.insert(0, str(ROOT))

import torch
import torch.nn as nn
import torch.optim as optim
import glob
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset

from models.flow import FlowMatching
from models.dit import DiT

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    
# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

film_conditioner.py STARTED
PyTorch version: 2.9.0+cpu
CUDA available: False
Using device: cpu


## Load Data with Custom DataLoader
Load a small sample of 10 data instances using the codec_dataloader module from the training package.

In [2]:
# Data paths
DATA_DIR = ROOT / "data" / "latent_data"
CLASSICAL_DIR = DATA_DIR / "latent_classical"
SYNTH_DIR = DATA_DIR / "latent_synth"

# Collect all file pairs
source_files = sorted(glob.glob(str(CLASSICAL_DIR / "*.pt")))
target_files = sorted(glob.glob(str(SYNTH_DIR / "*.pt")))

print(f"Found {len(source_files)} classical (source) files")
print(f"Found {len(target_files)} synth (target) files")

# Take only first 10 samples
sample_size = 10
source_files_sample = source_files[:sample_size]
target_files_sample = target_files[:sample_size]

print(f"\nUsing {len(source_files_sample)} source files")
print(f"Using {len(target_files_sample)} target files")

# Load data directly into memory
print("\nLoading latent data into memory...")
x0_list = []
x1_list = []

for src_file, tgt_file in zip(source_files_sample, target_files_sample):
    src_data = torch.load(src_file, map_location='cpu')
    tgt_data = torch.load(tgt_file, map_location='cpu')
    
    # Extract embeddings from the saved format (key 'z')
    if isinstance(src_data, dict):
        src_emb = src_data.get('z', src_data.get('embeddings', src_data.get('latents')))
    else:
        src_emb = src_data
    
    if isinstance(tgt_data, dict):
        tgt_emb = tgt_data.get('z', tgt_data.get('embeddings', tgt_data.get('latents')))
    else:
        tgt_emb = tgt_data
    
    # Remove batch dimension if present (from [1, T, D] to [T, D])
    if src_emb.dim() == 3 and src_emb.size(0) == 1:
        src_emb = src_emb.squeeze(0)
    if tgt_emb.dim() == 3 and tgt_emb.size(0) == 1:
        tgt_emb = tgt_emb.squeeze(0)
    
    x0_list.append(src_emb)
    x1_list.append(tgt_emb)

print(f"Loaded {len(x0_list)} pairs")

# Check sample shapes and find max sequence length
print(f"\nSample shapes:")
for i in range(min(3, len(x0_list))):
    print(f"  Sample {i}: x0 shape {x0_list[i].shape}, x1 shape {x1_list[i].shape}")

# Find max sequence length
max_seq_len = max(max(x.size(0) for x in x0_list), max(x.size(0) for x in x1_list))
print(f"\nMax sequence length: {max_seq_len}")

# Pad all sequences to the same length
def pad_sequence(seq, target_len):
    """Pad sequence to target length"""
    if seq.size(0) < target_len:
        pad_len = target_len - seq.size(0)
        # Pad with zeros at the end
        seq = torch.nn.functional.pad(seq, (0, 0, 0, pad_len), mode='constant', value=0)
    return seq

x0_list_padded = [pad_sequence(x, max_seq_len) for x in x0_list]
x1_list_padded = [pad_sequence(x, max_seq_len) for x in x1_list]

print(f"Padded all sequences to length {max_seq_len}")
print(f"  x0_padded[0] shape: {x0_list_padded[0].shape}, dtype: {x0_list_padded[0].dtype}")
print(f"  x1_padded[0] shape: {x1_list_padded[0].shape}, dtype: {x1_list_padded[0].dtype}")

Found 413 classical (source) files
Found 413 synth (target) files

Using 10 source files
Using 10 target files

Loading latent data into memory...
Loaded 10 pairs

Sample shapes:
  Sample 0: x0 shape torch.Size([2585, 1024]), x1 shape torch.Size([2794, 1024])
  Sample 1: x0 shape torch.Size([2585, 1024]), x1 shape torch.Size([2793, 1024])
  Sample 2: x0 shape torch.Size([2582, 1024]), x1 shape torch.Size([2790, 1024])

Max sequence length: 2794
Padded all sequences to length 2794
  x0_padded[0] shape: torch.Size([2794, 1024]), dtype: torch.float32
  x1_padded[0] shape: torch.Size([2794, 1024]), dtype: torch.float32


In [3]:
# Create simple batches from loaded data
batch_size = 10  # Use entire dataset as one batch (critical for overfitting)

print(f"Creating batches with batch_size={batch_size}")
print(f"Number of samples: {len(x0_list_padded)}")

# Stack all samples into tensors
x0_all = torch.stack(x0_list_padded, dim=0)
x1_all = torch.stack(x1_list_padded, dim=0)

print(f"\nBatch shapes:")
print(f"  x0_all: {x0_all.shape}")
print(f"  x1_all: {x1_all.shape}")

# For simplicity, we'll use the entire dataset as one batch
# Create a simple list of batches
batch_data = [(x0_all, x1_all)]

print(f"Created {len(batch_data)} batch(es)")
print(f"Batch 0: x0 shape {batch_data[0][0].shape}, x1 shape {batch_data[0][1].shape}")

Creating batches with batch_size=10
Number of samples: 10

Batch shapes:
  x0_all: torch.Size([10, 2794, 1024])
  x1_all: torch.Size([10, 2794, 1024])
Created 1 batch(es)
Batch 0: x0 shape torch.Size([10, 2794, 1024]), x1 shape torch.Size([10, 2794, 1024])


## Initialize Model and Training Components
Initialize the DiT model (the flow matching backbone) and wrap it in the FlowMatching loss module.

In [4]:
# Initialize DiT model
print("Initializing DiT model...")
dit_model = DiT()
dit_model = dit_model.to(DEVICE)
print(f"DiT model parameters: {sum(p.numel() for p in dit_model.parameters()):,}")

# Wrap in FlowMatching loss module
flow_matching = FlowMatching(dit_model)
flow_matching = flow_matching.to(DEVICE)

print("Models initialized successfully")

Initializing DiT model...
DiT model parameters: 6,835,968
Models initialized successfully


## Configure Training Parameters
Set training hyperparameters including learning rate, number of epochs, and optimizer configuration.

In [5]:
# Training configuration - OPTIMIZED FOR OVERFITTING
num_epochs = 50  # Increased from 5 - need many epochs to memorize
learning_rate = 1e-3  # Increased from 1e-4 - accelerate convergence
weight_decay = 0.0  # CRITICAL: Disable L2 regularization
batch_size = 10  # Use entire dataset as single batch (was 2)

# Recreate optimizer without weight decay
optimizer = optim.Adam(
    flow_matching.parameters(),
    lr=learning_rate
)

# Remove learning rate scheduler for overfitting test
# (steady learning is better for memorization than scheduling)

print(f"Training Configuration (OVERFITTING MODE):")
print(f"  Number of epochs: {num_epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Batch size: {batch_size}")
print(f"  Weight decay: {weight_decay} (DISABLED for overfitting)")
print(f"  Optimizer: Adam (without weight decay)")
print(f"  Device: {DEVICE}")
print(f"\nTarget: Model should memorize all {sample_size} samples")

Training Configuration (OVERFITTING MODE):
  Number of epochs: 50
  Learning rate: 0.001
  Batch size: 10
  Weight decay: 0.0 (DISABLED for overfitting)
  Optimizer: Adam (without weight decay)
  Device: cpu

Target: Model should memorize all 10 samples


## Create Training Loop
Implement the main training loop that iterates through epochs and batches, computing loss and updating model weights.

In [ ]:
# Training loop - OPTIMIZED FOR OVERFITTING
epoch_losses = []
batch_losses = []
gradient_norms = []

# Target loss threshold for early stopping
target_loss = 1e-4

print("\n" + "="*70)
print("OVERFITTING TEST: Training model to memorize 10 samples")
print("="*70 + "\n")

for epoch in range(num_epochs):
    epoch_loss = 0.0
    num_batches = 0
    
    # Training phase
    flow_matching.train()
    
    # Use the batch data we created
    for batch_idx, (x0_batch, x1_batch) in enumerate(batch_data):
        # Move to device
        x0_batch = x0_batch.to(DEVICE)
        x1_batch = x1_batch.to(DEVICE)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Create dummy genre_ids
        batch_size_actual = x0_batch.size(0)
        genre_ids = torch.zeros(batch_size_actual, dtype=torch.long, device=DEVICE)
        
        # Forward pass - compute flow matching loss
        loss = flow_matching.compute_loss(x0_batch, x1_batch, genre_ids)
        
        # Backward pass
        loss.backward()
        
        # OPTIONAL: Increase gradient clipping threshold (or comment out to disable)
        torch.nn.utils.clip_grad_norm_(flow_matching.parameters(), max_norm=5.0)
        
        # Monitor gradient norm every 50 epochs
        if epoch % 50 == 0:
            total_norm = 0.0
            for p in flow_matching.parameters():
                if p.grad is not None:
                    param_norm = p.grad.data.norm(2)
                    total_norm += param_norm.item() ** 2
            total_norm = total_norm ** 0.5
            gradient_norms.append(total_norm)
            
            if total_norm < 1e-6:
                print(f"  ⚠️ Warning at epoch {epoch+1}: Vanishing gradients (norm={total_norm:.2e})")
            elif total_norm > 100:
                print(f"  ⚠️ Warning at epoch {epoch+1}: Exploding gradients (norm={total_norm:.2e})")
        
        # Optimizer step
        optimizer.step()
        
        # Track loss
        batch_loss = loss.item()
        epoch_loss += batch_loss
        batch_losses.append(batch_loss)
        num_batches += 1
    
    # Average loss for epoch
    avg_epoch_loss = epoch_loss / num_batches
    epoch_losses.append(avg_epoch_loss)
    
    # Log progress every 10 epochs or when loss is very small
    if (epoch + 1) % 10 == 0 or avg_epoch_loss < target_loss:
        print(f"Epoch {epoch+1:3d}/{num_epochs} | Loss: {avg_epoch_loss:.8f}")
    
    # Early stopping when overfitting is achieved
    if avg_epoch_loss < target_loss:
        print(f"\n✓ SUCCESS! Model overfitted at epoch {epoch+1}")
        print(f"  Final loss: {avg_epoch_loss:.8f}")
        print(f"  Loss reduction: {epoch_losses[0] - avg_epoch_loss:.6f}")
        break

print("\n" + "="*70)
print("Training Completed!")
print(f"Total epochs run: {len(epoch_losses)}")
print(f"Initial loss: {epoch_losses[0]:.8f}")
print(f"Final loss: {epoch_losses[-1]:.8f}")
print("="*70)


OVERFITTING TEST: Training model to memorize 10 samples



## Monitor Training Progress
Track and display training metrics such as loss per epoch and visualize the training progression.

In [ ]:
# Print comprehensive training summary
print("\n" + "="*70)
print("TRAINING SUMMARY - OVERFITTING TEST")
print("="*70)

print(f"\nLoss Statistics:")
print(f"  Initial loss: {epoch_losses[0]:.8f}")
print(f"  Final loss:   {epoch_losses[-1]:.8f}")
print(f"  Loss reduction: {(epoch_losses[0] - epoch_losses[-1]):.8f}")

if epoch_losses[0] > 0:
    improvement = ((epoch_losses[0] - epoch_losses[-1]) / epoch_losses[0]) * 100
    print(f"  Improvement: {improvement:.2f}%")

print(f"\nConvergence Status:")
if epoch_losses[-1] < 1e-4:
    print(f"  ✓ EXCELLENT: Loss is < 1e-4 (strong overfitting)")
elif epoch_losses[-1] < 1e-3:
    print(f"  ✓ GOOD: Loss is < 1e-3 (moderate overfitting)")
elif epoch_losses[-1] < 0.01:
    print(f"  ◐ OK: Loss is < 0.01 (weak overfitting)")
else:
    print(f"  ✗ POOR: Loss is >= 0.01 (model not memorizing)")
    print(f"    → Try increasing epochs (current: {len(epoch_losses)})")
    print(f"    → Try increasing learning rate")
    print(f"    → Check that dropout=0.0 in DiT model")

print(f"\nGradient Statistics:")
if gradient_norms:
    print(f"  Average gradient norm: {sum(gradient_norms)/len(gradient_norms):.6f}")
    print(f"  Min gradient norm: {min(gradient_norms):.2e}")
    print(f"  Max gradient norm: {max(gradient_norms):.2e}")
else:
    print(f"  Gradient monitoring not enabled (runs every 50 epochs)")

print("="*70)

In [ ]:
# Visualize training progress
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Epoch losses (log scale)
axes[0, 0].semilogy(range(1, len(epoch_losses) + 1), epoch_losses, 'b-o', linewidth=2, markersize=6)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Loss (log scale)', fontsize=12)
axes[0, 0].set_title('Epoch Loss - Log Scale', fontsize=14, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3, which='both')
axes[0, 0].axhline(y=1e-4, color='r', linestyle='--', label='Target (1e-4)')
axes[0, 0].legend()

# Plot 2: Epoch losses (linear scale)
axes[0, 1].plot(range(1, len(epoch_losses) + 1), epoch_losses, 'b-o', linewidth=2, markersize=6)
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Loss', fontsize=12)
axes[0, 1].set_title('Epoch Loss - Linear Scale', fontsize=14, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axhline(y=1e-4, color='r', linestyle='--', label='Target (1e-4)')
axes[0, 1].legend()

# Plot 3: All batch losses
axes[1, 0].semilogy(batch_losses, 'g-', linewidth=1, alpha=0.7)
axes[1, 0].set_xlabel('Batch', fontsize=12)
axes[1, 0].set_ylabel('Loss (log scale)', fontsize=12)
axes[1, 0].set_title('Loss per Batch (All Epochs)', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, which='both')

# Plot 4: Gradient norms
if gradient_norms:
    epochs_monitored = [i * 50 for i in range(len(gradient_norms))]
    axes[1, 1].semilogy(epochs_monitored, gradient_norms, 'r-s', linewidth=2, markersize=6)
    axes[1, 1].set_xlabel('Epoch', fontsize=12)
    axes[1, 1].set_ylabel('Gradient Norm (log scale)', fontsize=12)
    axes[1, 1].set_title('Gradient Norm Monitoring', fontsize=14, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3, which='both')
    axes[1, 1].axhline(y=1e-6, color='orange', linestyle='--', label='Vanishing threshold')
    axes[1, 1].legend()
else:
    axes[1, 1].text(0.5, 0.5, 'Gradient monitoring\nnot enabled', 
                    ha='center', va='center', fontsize=14, transform=axes[1, 1].transAxes)
    axes[1, 1].set_title('Gradient Norm (Not Available)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(str(ROOT / "outputs" / "training_progress_overfitting.png"), dpi=150, bbox_inches='tight')
print("✓ Training progress plot saved to outputs/training_progress_overfitting.png")
plt.show()

## Save Model Checkpoint
Save the trained model checkpoint after training completes for later inference or fine-tuning.

In [ ]:
# Create checkpoint directory
checkpoint_dir = ROOT / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Save model checkpoint
checkpoint_path = checkpoint_dir / "small_sample_training.pt"

checkpoint = {
    'epoch': num_epochs,
    'model_state_dict': flow_matching.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'losses': epoch_losses,
    'batch_losses': batch_losses,
    'config': {
        'num_epochs': num_epochs,
        'learning_rate': learning_rate,
        'batch_size': batch_size,
        'num_samples': sample_size,
        'weight_decay': weight_decay
    }
}

torch.save(checkpoint, checkpoint_path)
print(f"Checkpoint saved to: {checkpoint_path}")
print(f"Checkpoint size: {checkpoint_path.stat().st_size / (1024*1024):.2f} MB")

In [ ]:
# VERIFICATION TEST: Confirm model successfully memorized training data
print("\n" + "="*70)
print("MEMORIZATION VERIFICATION TEST")
print("="*70 + "\n")

import torch.nn.functional as F

flow_matching.eval()
reconstruction_errors = []

print("Testing reconstruction of each training sample:\n")

with torch.no_grad():
    for idx in range(len(x0_list_padded)):
        x0 = x0_list_padded[idx]
        x1 = x1_list_padded[idx]
        x0_batch = x0.unsqueeze(0).to(DEVICE)  # Add batch dimension
        x1_batch = x1.unsqueeze(0).to(DEVICE)
        
        # Create genre_ids
        genre_ids = torch.tensor([0], dtype=torch.long, device=DEVICE)
        
        # Generate transformed audio using Euler sampler
        try:
            x_generated = flow_matching.sample_euler(
                x0_batch, 
                genre_ids, 
                num_steps=50
            )
            
            # Compute reconstruction error
            error = F.mse_loss(x_generated, x1_batch).item()
            reconstruction_errors.append(error)
            
            status = "✓" if error < 0.01 else "◐" if error < 0.1 else "✗"
            print(f"  Sample {idx:2d}: Reconstruction error = {error:.8f} {status}")
        except Exception as e:
            print(f"  Sample {idx:2d}: Error during inference - {e}")
            reconstruction_errors.append(float('nan'))

print("\n" + "="*70)
print("MEMORIZATION RESULTS:")
print("="*70)

avg_error = np.nanmean(reconstruction_errors) if reconstruction_errors else float('nan')
min_error = np.nanmin(reconstruction_errors) if reconstruction_errors else float('nan')
max_error = np.nanmax(reconstruction_errors) if reconstruction_errors else float('nan')

print(f"\nAverage reconstruction error: {avg_error:.8f}")
print(f"Min error: {min_error:.8f}")
print(f"Max error: {max_error:.8f}")

print(f"\nOVERFITTING VERDICT:")
if avg_error < 0.001:
    print(f"  ✓✓✓ EXCELLENT: Model perfectly memorized training data!")
    print(f"      Average error: {avg_error:.2e}")
elif avg_error < 0.01:
    print(f"  ✓✓ GOOD: Model substantially memorized training data")
    print(f"     Average error: {avg_error:.2e}")
elif avg_error < 0.1:
    print(f"  ✓ MODERATE: Model partially memorized training data")
    print(f"    Average error: {avg_error:.2e}")
else:
    print(f"  ✗ FAILED: Model did not memorize training data")
    print(f"    Average error: {avg_error:.2e}")
    print(f"\n  Debugging suggestions:")
    print(f"    1. Check that dropout=0.0 is set in DiT model")
    print(f"    2. Verify weight_decay=0.0 in optimizer")
    print(f"    3. Try increasing epochs (tested: {len(epoch_losses)})")
    print(f"    4. Try increasing learning_rate (tested: {learning_rate})")
    print(f"    5. Check for NaN in loss values")

print("="*70)